Task 1: Topic Classification using BERT

Dataset: AG News
Goal: Train a BERT model to classify news headlines into 4 categories

Step 1: Install Required Libraries
Install Hugging Face Transformers, Datasets, PyTorch, scikit-learn, and visualization libraries for data exploration.

In [ ]:
#  Install required libraries
!pip install transformers datasets torch scikit-learn accelerate -q

Step 2: Imports

In [ ]:
#Imports
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score

Step 3: Load and Explore Dataset

Description: Load the AG News dataset and explore its structure, sample entries, and class distribution.

In [ ]:
# Load AG News dataset
dataset = load_dataset("ag_news")

print(dataset)
print("\nSample entry:", dataset["train"][0])
print("\nLabel names:", dataset["train"].features["label"].names)

# AG News has 4 classes:
# 0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech
label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print(f"\nNumber of classes: {num_labels}")

Step 4: Tokenization / Preprocessing
Description: Tokenize the text using BERT tokenizer. Limit sequence length to 128 tokens. Padding will be handled dynamically during training.

In [ ]:
# Tokenize
MODEL_NAME = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,      # 128 is enough for news headlines
        padding=False        # DataCollator handles padding dynamically
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Rename label column to what Trainer expects
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Tokenization complete!")
print("Train size:", len(tokenized_dataset["train"]))
print("Test size:", len(tokenized_dataset["test"]))

Step 5: Subsample Dataset
Description: For faster training on Colab, we can subsample the dataset. Remove this step for full dataset training.

In [ ]:
#Subsample for Colab speed (optional — remove for full training)
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(5000))
eval_dataset  = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))

print(f"Using {len(train_dataset)} train / {len(eval_dataset)} eval samples")

Step 6: Load BERT Model
Description: Load pre-trained BERT model for sequence classification. Map the label IDs to class names.

In [ ]:
#  Load BERT for sequence classification
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={i: label_names[i] for i in range(num_labels)},
    label2id={label_names[i]: i for i in range(num_labels)}
)

print("Model loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Step 7: Define Evaluation Metrics
Description: Accuracy and weighted F1-score are used to evaluate performance.

In [ ]:
# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": round(acc, 4),
        "f1_weighted": round(f1, 4)
    }

Step 8: Training Configuration
Description: Configure hyperparameters for model training including learning rate, batch size, number of epochs, and evaluation strategy.

In [ ]:
# Cell 8: Training configuration
from transformers import TrainingArguments
import transformers

print(f"Transformers version: {transformers.__version__}")

training_args = TrainingArguments(
    output_dir="./bert-ag-news",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",        # renamed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=50,
    fp16=True,
    report_to="none"
)

print("TrainingArguments created successfully!")

Step 9: Initialize Trainer & Train
Description: Use Hugging Face Trainer to train BERT on the dataset.

In [ ]:
# Initialize Trainer and train
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,      # renamed from 'tokenizer'
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Step 10: Evaluate Model

Description: Evaluate the trained model on the test set.

In [ ]:
# Final evaluation
results = trainer.evaluate()

print("\n" + "="*40)
print("FINAL EVALUATION RESULTS")
print("="*40)
print(f"Accuracy  : {results['eval_accuracy']:.4f}")
print(f"F1 Score  : {results['eval_f1_weighted']:.4f}")
print(f"Eval Loss : {results['eval_loss']:.4f}")

Step 11: Inference on New Headlines
Description: Use the trained model to classify unseen news headlines.

In [ ]:
# Inference on new headlines
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_headlines = [
    "Stock markets surge as inflation fears ease",
    "Champions League final: Real Madrid defeats Liverpool",
    "NASA launches new Mars exploration rover",
    "UN Security Council meets over Ukraine conflict"
]

print("\nPredictions:")
print("-" * 50)
for headline in test_headlines:
    result = classifier(headline)[0]
    print(f"Text  : {headline}")
    print(f"Label : {result['label']}  |  Score: {result['score']:.4f}")
    print()

Step 12: Save Model
Description: Save the trained model and tokenizer for future use.

In [ ]:
# Save model to Google Drive
model.save_pretrained("./bert-ag-news-final")
tokenizer.save_pretrained("./bert-ag-news-final")
print("Model saved!")

Step 13: Summary & Insights

### Summary & Insights
- BERT achieved high accuracy on the AG News dataset (~90%+ on subsample).
- Weighted F1-score confirms balanced performance across all classes.
- Transformer models effectively capture contextual meaning of text.
- Class distribution is slightly imbalanced; full dataset training is recommended.

### Limitations
- Training is computationally expensive.
- Subsampled dataset may not fully represent all classes.

### Future Improvements
- Train on the full dataset (~120k samples).
- Perform hyperparameter tuning (learning rate, batch size, epochs).
- Explore advanced models like RoBERTa, DistilBERT, or domain-specific BERT variants.